In [ ]:
!pip -q install -U langchain langchain-openai langchain-core openai
!pip -q install pandas==2.2.2 "scikit-learn>=1.2,<1.9"

In [ ]:
# =========================================================
# 0. API key for Colab
# =========================================================

import os

# For DeepSeek experiments
DEEPSEEK_API_KEY = "key"
os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY

# For OpenAI/GPT experiments, uncomment and set this:
# OPENAI_API_KEY = "your_openai_api_key_here"
# os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


# =========================================================
# 1. Imports
# =========================================================

import json
import time
import re
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd

from pydantic import BaseModel, Field

from sklearn.base import clone
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeClassifier
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    RandomForestClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI


# =========================================================
# 2. General settings
# =========================================================

# Choose: "DeepSeek" or "OpenAI"
LLM_PROVIDER = "DeepSeek"

# DeepSeek settings
DEEPSEEK_MODEL_ID = "deepseek-v4-pro"
DEEPSEEK_DISPLAY_NAME = "DeepSeek-V4-Pro"
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_THINKING_TYPE = "disabled"

# OpenAI settings
OPENAI_MODEL_ID = "gpt-4o"
OPENAI_DISPLAY_NAME = "GPT-4o"

TEMPERATURE = 0.0

# Change this path to any previous config file.
CONFIG_PATH = "/content/multiclass_random_forest_classifier_experiment_config.json"

DATA_DIR = "/content"
OUTPUT_DIR = "/content/langchain_generic_agent_outputs"

# Set to 5 for final repeated-run experiments.
NUM_RUNS = 1

SAVE_MESSAGE_TRACE = False

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================================================
# 3. Config loading
# =========================================================

def load_experiment_configs(config_path: str) -> List[Dict[str, Any]]:
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Config file not found: {config_path}")

    with open(config_path, "r", encoding="utf-8") as f:
        configs = json.load(f)

    if not isinstance(configs, list):
        raise ValueError("The config JSON must contain a list of experiment configs.")

    return configs


def safe_name(name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_\-]+", "_", str(name).strip())


def get_dataset_path(config: Dict[str, Any]) -> str:
    if config.get("dataset_path"):
        return config["dataset_path"]
    return os.path.join(DATA_DIR, config["file_name"])


def get_model_name(config: Dict[str, Any]) -> str:
    model_name = (
        config.get("configured_model_name")
        or config.get("model_name")
        or config.get("configured_model")
    )

    if not model_name:
        raise ValueError(f"Missing model name in config: {config.get('dataset_name')}")

    return model_name


def infer_task_type(config: Dict[str, Any]) -> str:
    if config.get("task_type"):
        return config["task_type"]

    model_name = get_model_name(config)

    if model_name in [
        "linear_regression",
        "random_forest_regressor",
        "gradient_boosting_regressor",
    ]:
        return "regression"

    if model_name in [
        "logistic_regression",
        "lda_classifier",
        "ridge_classifier",
    ]:
        return "binary_classification"

    if model_name in [
        "decision_tree_classifier",
        "random_forest_classifier",
    ]:
        return "multiclass_classification"

    raise ValueError(f"Cannot infer task type from model name: {model_name}")


MODEL_DISPLAY_NAMES = {
    "linear_regression": "Linear Regression",
    "random_forest_regressor": "Random Forest Regressor",
    "gradient_boosting_regressor": "Gradient Boosting Regressor",
    "logistic_regression": "Logistic Regression",
    "lda_classifier": "Linear Discriminant Analysis",
    "ridge_classifier": "Ridge Classifier",
    "decision_tree_classifier": "Decision Tree Classifier",
    "random_forest_classifier": "Random Forest Classifier",
}


# =========================================================
# 4. Dataset loading
# =========================================================

def load_dataset(config: Dict[str, Any]) -> pd.DataFrame:
    dataset_path = get_dataset_path(config)
    sep = config.get("csv_sep", ",")

    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Dataset not found: {dataset_path}")

    return pd.read_csv(dataset_path, sep=sep)


def validate_columns(
    df: pd.DataFrame,
    x_columns: List[str],
    y_column: str,
    dataset_name: str
) -> None:
    missing = [c for c in x_columns + [y_column] if c not in df.columns]

    if missing:
        raise ValueError(
            f"[{dataset_name}] Missing columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )


def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def split_feature_types(df: pd.DataFrame, x_columns: List[str]) -> Dict[str, List[str]]:
    numeric_columns = []
    categorical_columns = []

    for col in x_columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_columns.append(col)
        else:
            categorical_columns.append(col)

    return {
        "numeric_columns": numeric_columns,
        "categorical_columns": categorical_columns,
    }


def clean_feature_name(name: str) -> str:
    name = str(name)

    if name.startswith("num__"):
        return name.replace("num__", "", 1)

    if name.startswith("cat__"):
        return name.replace("cat__", "", 1)

    return name


def build_preprocessor(
    X: pd.DataFrame,
    x_columns: List[str]
) -> Tuple[ColumnTransformer, Dict[str, List[str]]]:
    feature_types = split_feature_types(X, x_columns)

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, feature_types["numeric_columns"]),
            ("cat", categorical_transformer, feature_types["categorical_columns"]),
        ],
        remainder="drop",
    )

    return preprocessor, feature_types


def build_model(model_name: str, task_type: str):
    if task_type == "regression":
        if model_name == "linear_regression":
            return LinearRegression()

        if model_name == "random_forest_regressor":
            return RandomForestRegressor(
                n_estimators=100,
                random_state=42,
            )

        if model_name == "gradient_boosting_regressor":
            return GradientBoostingRegressor(
                random_state=42,
            )

    if task_type in ["binary_classification", "multiclass_classification"]:
        if model_name == "logistic_regression":
            return LogisticRegression(
                max_iter=2000,
                random_state=42,
            )

        if model_name == "lda_classifier":
            return LinearDiscriminantAnalysis()

        if model_name == "ridge_classifier":
            return RidgeClassifier(
                random_state=42,
            )

        if model_name == "decision_tree_classifier":
            return DecisionTreeClassifier(
                random_state=42,
            )

        if model_name == "random_forest_classifier":
            return RandomForestClassifier(
                n_estimators=100,
                random_state=42,
            )

    raise ValueError(f"Unsupported model '{model_name}' for task type '{task_type}'.")


def build_pipeline(
    X: pd.DataFrame,
    x_columns: List[str],
    model_name: str,
    task_type: str
) -> Tuple[Pipeline, Dict[str, List[str]]]:
    preprocessor, feature_types = build_preprocessor(X, x_columns)
    model = build_model(model_name, task_type)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    return pipeline, feature_types


def get_feature_names_from_pipeline(pipeline: Pipeline) -> List[str]:
    preprocessor = pipeline.named_steps["preprocessor"]
    raw_feature_names = preprocessor.get_feature_names_out()

    return [
        clean_feature_name(name)
        for name in raw_feature_names
    ]


def safe_train_test_split_for_classification(X, y):
    try:
        counts = pd.Series(y).value_counts(dropna=False)
        if len(counts) > 1 and counts.min() >= 2:
            return train_test_split(
                X,
                y,
                test_size=0.2,
                random_state=42,
                stratify=y,
            )
    except Exception:
        pass

    return train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=None,
    )


def compute_train_test_metrics(
    base_pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    task_type: str
) -> Dict[str, Any]:
    if len(X) < 5:
        return {
            "train_metric": None,
            "test_metric": None,
            "metric_name": None,
            "split_note": "Too few rows for train/test split.",
        }

    try:
        if task_type == "regression":
            X_train, X_test, y_train, y_test = train_test_split(
                X,
                y,
                test_size=0.2,
                random_state=42,
            )

            split_pipeline = clone(base_pipeline)
            split_pipeline.fit(X_train, y_train)

            train_pred = split_pipeline.predict(X_train)
            test_pred = split_pipeline.predict(X_test)

            return {
                "train_metric": float(r2_score(y_train, train_pred)),
                "test_metric": float(r2_score(y_test, test_pred)),
                "metric_name": "r_squared",
                "split_note": "Train/test split uses test_size=0.2 and random_state=42.",
            }

        X_train, X_test, y_train, y_test = safe_train_test_split_for_classification(X, y)

        split_pipeline = clone(base_pipeline)
        split_pipeline.fit(X_train, y_train)

        train_pred = split_pipeline.predict(X_train)
        test_pred = split_pipeline.predict(X_test)

        return {
            "train_metric": float(accuracy_score(y_train, train_pred)),
            "test_metric": float(accuracy_score(y_test, test_pred)),
            "metric_name": "accuracy",
            "split_note": "Train/test split uses test_size=0.2 and random_state=42; stratification is used when feasible.",
        }

    except Exception as exc:
        return {
            "train_metric": None,
            "test_metric": None,
            "metric_name": None,
            "split_note": f"Train/test metric computation failed: {str(exc)}",
        }


# =========================================================
# 5. Generic configured analysis
# =========================================================

def summarize_regression_model(
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    feature_names: List[str],
    model_name: str
) -> Dict[str, Any]:
    trained_model = pipeline.named_steps["model"]
    y_pred = pipeline.predict(X)

    r_squared = float(r2_score(y, y_pred))

    if hasattr(trained_model, "coef_"):
        values = np.asarray(trained_model.coef_, dtype=float).ravel()
        evidence_type = "coefficients"
    elif hasattr(trained_model, "feature_importances_"):
        values = np.asarray(trained_model.feature_importances_, dtype=float).ravel()
        evidence_type = "feature_importances"
    else:
        values = np.zeros(len(feature_names), dtype=float)
        evidence_type = "unavailable"

    if len(values) < len(feature_names):
        values = np.pad(values, (0, len(feature_names) - len(values)))
    elif len(values) > len(feature_names):
        values = values[:len(feature_names)]

    signed_values = {
        str(feature): float(value)
        for feature, value in zip(feature_names, values)
    }

    abs_values = {
        str(feature): abs(float(value))
        for feature, value in zip(feature_names, values)
    }

    most_influential_feature = max(abs_values, key=abs_values.get) if abs_values else None

    top_features = sorted(
        abs_values.items(),
        key=lambda x: x[1],
        reverse=True,
    )[:10]

    return {
        "performance": {
            "r_squared_full_data": r_squared,
            "metric_note": "The full-data R-squared is calculated on the same data used for fitting.",
        },
        "evidence_type": evidence_type,
        "feature_effects": signed_values,
        "absolute_feature_effects": abs_values,
        "most_influential_feature": most_influential_feature,
        "most_influential_value": abs_values.get(most_influential_feature) if most_influential_feature else None,
        "top_10_features_by_absolute_effect": [
            {
                "feature": feature,
                "absolute_effect": value,
                "signed_effect": signed_values.get(feature),
            }
            for feature, value in top_features
        ],
    }


def summarize_classification_model(
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    feature_names: List[str],
    model_name: str
) -> Dict[str, Any]:
    trained_model = pipeline.named_steps["model"]
    y_pred = pipeline.predict(X)

    accuracy = float(accuracy_score(y, y_pred))

    classes = list(getattr(trained_model, "classes_", sorted(pd.Series(y).dropna().unique())))
    class_distribution = pd.Series(y).value_counts(dropna=False).to_dict()
    class_distribution = {str(k): int(v) for k, v in class_distribution.items()}

    if hasattr(trained_model, "coef_"):
        coef_matrix = np.asarray(trained_model.coef_, dtype=float)

        if coef_matrix.ndim == 1:
            coef_matrix = coef_matrix.reshape(1, -1)

        if coef_matrix.shape[0] == 1:
            class_labels = [str(classes[-1]) if classes else "positive_class"]
        else:
            class_labels = [str(c) for c in classes[:coef_matrix.shape[0]]]

        if coef_matrix.shape[1] < len(feature_names):
            coef_matrix = np.pad(
                coef_matrix,
                ((0, 0), (0, len(feature_names) - coef_matrix.shape[1])),
            )
        elif coef_matrix.shape[1] > len(feature_names):
            coef_matrix = coef_matrix[:, :len(feature_names)]

        coefficient_details = {
            class_labels[class_index]: {
                str(feature): float(coef_matrix[class_index][feature_index])
                for feature_index, feature in enumerate(feature_names)
            }
            for class_index in range(coef_matrix.shape[0])
        }

        avg_abs_effects = {
            str(feature): float(np.mean(np.abs(coef_matrix[:, feature_index])))
            for feature_index, feature in enumerate(feature_names)
        }

        evidence_type = "coefficients"

    elif hasattr(trained_model, "feature_importances_"):
        importances = np.asarray(trained_model.feature_importances_, dtype=float).ravel()

        if len(importances) < len(feature_names):
            importances = np.pad(importances, (0, len(feature_names) - len(importances)))
        elif len(importances) > len(feature_names):
            importances = importances[:len(feature_names)]

        coefficient_details = {
            str(feature): float(value)
            for feature, value in zip(feature_names, importances)
        }

        avg_abs_effects = {
            str(feature): float(value)
            for feature, value in zip(feature_names, importances)
        }

        evidence_type = "feature_importances"

    else:
        coefficient_details = {}
        avg_abs_effects = {
            str(feature): 0.0
            for feature in feature_names
        }
        evidence_type = "unavailable"

    most_influential_feature = max(avg_abs_effects, key=avg_abs_effects.get) if avg_abs_effects else None

    top_features = sorted(
        avg_abs_effects.items(),
        key=lambda x: x[1],
        reverse=True,
    )[:10]

    return {
        "performance": {
            "accuracy_full_data": accuracy,
            "metric_note": "The full-data accuracy is calculated on the same data used for fitting.",
        },
        "classes": [str(c) for c in classes],
        "class_distribution": class_distribution,
        "evidence_type": evidence_type,
        "coefficient_or_importance_details": coefficient_details,
        "average_absolute_feature_effects": avg_abs_effects,
        "most_influential_feature": most_influential_feature,
        "most_influential_value": avg_abs_effects.get(most_influential_feature) if most_influential_feature else None,
        "top_10_features_by_absolute_effect": [
            {
                "feature": feature,
                "absolute_effect": value,
            }
            for feature, value in top_features
        ],
    }


def run_configured_analysis(config: Dict[str, Any]) -> Dict[str, Any]:
    dataset_name = config["dataset_name"]
    x_columns = config["x_columns"]
    y_column = config["y_column"]
    model_name = get_model_name(config)
    task_type = infer_task_type(config)

    df = load_dataset(config)
    validate_columns(df, x_columns, y_column, dataset_name)

    df = df[x_columns + [y_column]].copy()
    df = df.dropna(subset=[y_column])

    X = df[x_columns].copy()
    y = df[y_column].copy()

    if task_type == "regression":
        y_numeric = pd.to_numeric(y, errors="coerce")
        valid_mask = y_numeric.notna()
        X = X.loc[valid_mask].copy()
        y = y_numeric.loc[valid_mask].copy()

    pipeline, feature_types = build_pipeline(
        X=X,
        x_columns=x_columns,
        model_name=model_name,
        task_type=task_type,
    )

    pipeline.fit(X, y)

    feature_names = get_feature_names_from_pipeline(pipeline)

    split_metrics = compute_train_test_metrics(
        base_pipeline=pipeline,
        X=X,
        y=y,
        task_type=task_type,
    )

    if task_type == "regression":
        model_summary = summarize_regression_model(
            pipeline=pipeline,
            X=X,
            y=y,
            feature_names=feature_names,
            model_name=model_name,
        )
    else:
        model_summary = summarize_classification_model(
            pipeline=pipeline,
            X=X,
            y=y,
            feature_names=feature_names,
            model_name=model_name,
        )

    return {
        "task_type": task_type,
        "configured_model_name": model_name,
        "configured_model_display_name": MODEL_DISPLAY_NAMES.get(model_name, model_name),
        "dataset_name": dataset_name,
        "file_name": config.get("file_name"),
        "shape_after_dropping_missing_y": list(df.shape),
        "x_columns": x_columns,
        "y_column": y_column,
        "numeric_columns": feature_types["numeric_columns"],
        "categorical_columns": feature_types["categorical_columns"],
        "preprocessed_feature_names": feature_names,
        "preprocessing_note": (
            "Numeric predictors are median-imputed and standardized. "
            "Categorical predictors are most-frequent-imputed and one-hot encoded. "
            "Reported coefficients or feature importances correspond to preprocessed model features."
        ),
        "train_test_metrics": split_metrics,
        "model_summary": model_summary,
    }


# =========================================================
# 6. Token and runtime extraction
# =========================================================

def normalize_usage(usage: Optional[Dict[str, Any]]) -> Dict[str, int]:
    if not usage:
        return {
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
        }

    prompt_tokens = (
        usage.get("prompt_tokens")
        or usage.get("input_tokens")
        or usage.get("input_token_count")
        or 0
    )

    completion_tokens = (
        usage.get("completion_tokens")
        or usage.get("output_tokens")
        or usage.get("output_token_count")
        or 0
    )

    total_tokens = (
        usage.get("total_tokens")
        or usage.get("total_token_count")
        or 0
    )

    if not total_tokens:
        total_tokens = prompt_tokens + completion_tokens

    return {
        "prompt_tokens": int(prompt_tokens),
        "completion_tokens": int(completion_tokens),
        "total_tokens": int(total_tokens),
    }


def extract_message_usage(message: Any) -> Dict[str, int]:
    metadata = getattr(message, "response_metadata", None)

    if isinstance(metadata, dict):
        token_usage = metadata.get("token_usage")
        if token_usage:
            return normalize_usage(token_usage)

        usage = metadata.get("usage")
        if usage:
            return normalize_usage(usage)

    usage_metadata = getattr(message, "usage_metadata", None)

    if isinstance(usage_metadata, dict):
        return normalize_usage(usage_metadata)

    return normalize_usage(None)


def summarize_agent_run(result: Dict[str, Any], runtime_seconds: float) -> Dict[str, Any]:
    messages = result.get("messages", [])

    prompt_tokens = 0
    completion_tokens = 0
    total_tokens = 0
    llm_call_count = 0
    tool_call_count = 0

    for message in messages:
        usage = extract_message_usage(message)

        if usage["total_tokens"] > 0:
            llm_call_count += 1
            prompt_tokens += usage["prompt_tokens"]
            completion_tokens += usage["completion_tokens"]
            total_tokens += usage["total_tokens"]

        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            tool_call_count += len(tool_calls)

    return {
        "llm_call_count": llm_call_count,
        "tool_call_count": tool_call_count,
        "usage": {
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total_tokens,
        },
        "runtime_seconds": float(runtime_seconds),
    }


def make_json_safe(obj: Any) -> Any:
    try:
        json.dumps(obj, ensure_ascii=False)
        return obj
    except Exception:
        return str(obj)


def serialize_message(message: Any) -> Dict[str, Any]:
    message_type = getattr(message, "type", None) or message.__class__.__name__
    content = getattr(message, "content", "")

    serialized = {
        "type": message_type,
        "content": content,
    }

    name = getattr(message, "name", None)
    if name:
        serialized["name"] = name

    tool_calls = getattr(message, "tool_calls", None)
    if tool_calls:
        serialized["tool_calls"] = make_json_safe(tool_calls)

    usage = extract_message_usage(message)
    if usage["total_tokens"] > 0:
        serialized["usage"] = usage

    response_metadata = getattr(message, "response_metadata", None)
    if response_metadata:
        serialized["response_metadata"] = make_json_safe(response_metadata)

    return serialized


# =========================================================
# 7. Build LLM and LangChain agent
# =========================================================

def get_llm_metadata() -> Dict[str, Any]:
    if LLM_PROVIDER.lower() == "deepseek":
        return {
            "llm_provider": "DeepSeek",
            "model_display_name": DEEPSEEK_DISPLAY_NAME,
            "model_id": DEEPSEEK_MODEL_ID,
            "api_base_url": DEEPSEEK_BASE_URL,
            "deepseek_thinking_type": DEEPSEEK_THINKING_TYPE,
            "temperature": TEMPERATURE,
        }

    if LLM_PROVIDER.lower() == "openai":
        return {
            "llm_provider": "OpenAI",
            "model_display_name": OPENAI_DISPLAY_NAME,
            "model_id": OPENAI_MODEL_ID,
            "api_base_url": None,
            "deepseek_thinking_type": None,
            "temperature": TEMPERATURE,
        }

    raise ValueError(f"Unsupported LLM_PROVIDER: {LLM_PROVIDER}")


def build_llm() -> ChatOpenAI:
    if LLM_PROVIDER.lower() == "deepseek":
        if not os.environ.get("DEEPSEEK_API_KEY"):
            raise EnvironmentError("Please set DEEPSEEK_API_KEY.")

        common_kwargs = {
            "model": DEEPSEEK_MODEL_ID,
            "api_key": os.environ["DEEPSEEK_API_KEY"],
            "base_url": DEEPSEEK_BASE_URL,
            "temperature": TEMPERATURE,
        }

        extra_body = {
            "thinking": {
                "type": DEEPSEEK_THINKING_TYPE
            }
        }

        try:
            return ChatOpenAI(
                **common_kwargs,
                extra_body=extra_body,
            )
        except TypeError:
            try:
                return ChatOpenAI(
                    **common_kwargs,
                    model_kwargs={"extra_body": extra_body},
                )
            except TypeError:
                return ChatOpenAI(**common_kwargs)

    if LLM_PROVIDER.lower() == "openai":
        if not os.environ.get("OPENAI_API_KEY"):
            raise EnvironmentError("Please set OPENAI_API_KEY.")

        return ChatOpenAI(
            model=OPENAI_MODEL_ID,
            api_key=os.environ["OPENAI_API_KEY"],
            temperature=TEMPERATURE,
        )

    raise ValueError(f"Unsupported LLM_PROVIDER: {LLM_PROVIDER}")


class InspectDatasetInput(BaseModel):
    dataset_path: str = Field(description="Path to the CSV dataset file.")


class RunConfiguredAnalysisInput(BaseModel):
    dataset_path: str = Field(description="Path to the CSV dataset file.")


def build_agent(config: Dict[str, Any]):
    llm = build_llm()

    model_name = get_model_name(config)
    task_type = infer_task_type(config)

    @tool(args_schema=InspectDatasetInput)
    def inspect_dataset(dataset_path: str) -> str:
        """Inspect the dataset and return its shape, columns, and data types."""
        df = load_dataset(config)

        result = {
            "shape": list(df.shape),
            "columns": list(df.columns),
            "dtypes": {col: str(dtype) for col, dtype in df.dtypes.items()},
            "dataset_path_received": dataset_path,
        }

        return json.dumps(result, ensure_ascii=False, indent=2)

    @tool(args_schema=RunConfiguredAnalysisInput)
    def run_configured_analysis_tool(dataset_path: str) -> str:
        """Run the configured analysis for the current experiment."""
        result = run_configured_analysis(config)
        return json.dumps(result, ensure_ascii=False, indent=2)

    tools = [
        inspect_dataset,
        run_configured_analysis_tool,
    ]

    llm_meta = get_llm_metadata()

    system_prompt = f"""
You are a LangChain-based tool-using single-agent baseline for analytical textual reporting.

Your task is to answer one user question at a time about a structured dataset.

Current experiment settings:
- Task type: {task_type}
- X columns: {config["x_columns"]}
- Y column: {config["y_column"]}
- Configured model for this experiment: {model_name}
- Configured model display name: {MODEL_DISPLAY_NAMES.get(model_name, model_name)}
- LLM provider: {llm_meta["llm_provider"]}
- LLM model: {llm_meta["model_id"]}

""".strip()

    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=system_prompt,
    )

    return agent


# =========================================================
# 8. Agent query
# =========================================================

def ask_agent(
    agent: Any,
    config: Dict[str, Any],
    question: str,
) -> Dict[str, Any]:
    dataset_path = get_dataset_path(config)
    model_name = get_model_name(config)
    task_type = infer_task_type(config)

    prompt = f"""
Dataset path:
{dataset_path}

Dataset name:
{config["dataset_name"]}

Task type:
{task_type}

Configured model:
{model_name}

Background information:
{config["background_knowledge"]}

Question:
{question}

Please answer the question using the available tools when analytical evidence is needed.
""".strip()

    start_time = time.time()

    result = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }
    )

    runtime_seconds = time.time() - start_time

    messages = result.get("messages", [])
    final_message = messages[-1] if messages else None
    answer = getattr(final_message, "content", str(final_message)) if final_message else ""

    run_summary = summarize_agent_run(result, runtime_seconds)

    output = {
        "question": question,
        "answer": answer,
        "status": "success" if answer else "failure",
        "failure_reason": None if answer else "Empty final answer.",
        "llm_call_count": run_summary["llm_call_count"],
        "tool_call_count": run_summary["tool_call_count"],
        "usage": run_summary["usage"],
        "runtime_seconds": run_summary["runtime_seconds"],
    }

    if SAVE_MESSAGE_TRACE:
        output["messages_trace"] = [serialize_message(m) for m in messages]

    return output


# =========================================================
# 9. Summary helpers
# =========================================================

def summarize_dataset_output(output: Dict[str, Any]) -> Dict[str, Any]:
    total_llm_call_count = 0
    total_tool_call_count = 0
    total_prompt_tokens = 0
    total_completion_tokens = 0
    total_tokens = 0
    total_runtime_seconds = 0.0
    success_count = 0
    failure_count = 0

    for qa in output["questions_and_answers"]:
        if qa.get("status") == "success":
            success_count += 1
        else:
            failure_count += 1

        total_llm_call_count += qa.get("llm_call_count", 0) or 0
        total_tool_call_count += qa.get("tool_call_count", 0) or 0

        usage = qa.get("usage") or {}

        total_prompt_tokens += usage.get("prompt_tokens", 0) or 0
        total_completion_tokens += usage.get("completion_tokens", 0) or 0
        total_tokens += usage.get("total_tokens", 0) or 0

        total_runtime_seconds += qa.get("runtime_seconds", 0.0) or 0.0

    n = len(output["questions_and_answers"])

    return {
        "num_questions": n,
        "success_count": success_count,
        "failure_count": failure_count,
        "failure_rate": failure_count / n if n else None,
        "total_llm_call_count": total_llm_call_count,
        "total_tool_call_count": total_tool_call_count,
        "total_prompt_tokens": total_prompt_tokens,
        "total_completion_tokens": total_completion_tokens,
        "total_tokens": total_tokens,
        "total_runtime_seconds": total_runtime_seconds,
        "avg_llm_call_count_per_answer": total_llm_call_count / n if n else None,
        "avg_tool_call_count_per_answer": total_tool_call_count / n if n else None,
        "avg_total_tokens_per_answer": total_tokens / n if n else None,
        "avg_runtime_seconds_per_answer": total_runtime_seconds / n if n else None,
    }


def summarize_all_outputs(all_outputs: List[Dict[str, Any]]) -> Dict[str, Any]:
    total_questions = 0
    total_failures = 0
    total_llm_calls = 0
    total_tool_calls = 0
    total_prompt_tokens = 0
    total_completion_tokens = 0
    total_tokens = 0
    total_runtime = 0.0

    for output in all_outputs:
        summary = output.get("dataset_summary") or {}

        total_questions += summary.get("num_questions", 0) or 0
        total_failures += summary.get("failure_count", 0) or 0
        total_llm_calls += summary.get("total_llm_call_count", 0) or 0
        total_tool_calls += summary.get("total_tool_call_count", 0) or 0
        total_prompt_tokens += summary.get("total_prompt_tokens", 0) or 0
        total_completion_tokens += summary.get("total_completion_tokens", 0) or 0
        total_tokens += summary.get("total_tokens", 0) or 0
        total_runtime += summary.get("total_runtime_seconds", 0.0) or 0.0

    return {
        "num_dataset_runs": len(all_outputs),
        "total_questions": total_questions,
        "total_failures": total_failures,
        "overall_failure_rate": total_failures / total_questions if total_questions else None,
        "total_llm_call_count": total_llm_calls,
        "total_tool_call_count": total_tool_calls,
        "total_prompt_tokens": total_prompt_tokens,
        "total_completion_tokens": total_completion_tokens,
        "total_tokens": total_tokens,
        "total_runtime_seconds": total_runtime,
        "avg_llm_call_count_per_answer": total_llm_calls / total_questions if total_questions else None,
        "avg_tool_call_count_per_answer": total_tool_calls / total_questions if total_questions else None,
        "avg_total_tokens_per_answer": total_tokens / total_questions if total_questions else None,
        "avg_runtime_seconds_per_answer": total_runtime / total_questions if total_questions else None,
    }


def summarize_repeated_runs(all_outputs: List[Dict[str, Any]]) -> Dict[str, Any]:
    rows = []

    for output in all_outputs:
        summary = output.get("dataset_summary") or {}
        rows.append({
            "run_id": output.get("run_id"),
            "dataset_name": output.get("dataset_name"),
            "task_type": output.get("task_type"),
            "configured_model_name": output.get("configured_model_name"),
            "num_questions": summary.get("num_questions", 0),
            "failure_count": summary.get("failure_count", 0),
            "failure_rate": summary.get("failure_rate"),
            "total_llm_call_count": summary.get("total_llm_call_count", 0),
            "total_tool_call_count": summary.get("total_tool_call_count", 0),
            "total_prompt_tokens": summary.get("total_prompt_tokens", 0),
            "total_completion_tokens": summary.get("total_completion_tokens", 0),
            "total_tokens": summary.get("total_tokens", 0),
            "total_runtime_seconds": summary.get("total_runtime_seconds", 0.0),
            "avg_total_tokens_per_answer": summary.get("avg_total_tokens_per_answer"),
            "avg_runtime_seconds_per_answer": summary.get("avg_runtime_seconds_per_answer"),
        })

    if not rows:
        return {}

    df = pd.DataFrame(rows)

    numeric_cols = [
        "failure_count",
        "failure_rate",
        "total_llm_call_count",
        "total_tool_call_count",
        "total_prompt_tokens",
        "total_completion_tokens",
        "total_tokens",
        "total_runtime_seconds",
        "avg_total_tokens_per_answer",
        "avg_runtime_seconds_per_answer",
    ]

    repeated_summary = {}
    for col in numeric_cols:
        if col in df.columns:
            repeated_summary[col] = {
                "mean": float(df[col].mean()),
                "std": float(df[col].std(ddof=1)) if len(df[col].dropna()) > 1 else 0.0,
                "min": float(df[col].min()),
                "max": float(df[col].max()),
            }

    return repeated_summary


# =========================================================
# 10. Run one dataset
# =========================================================

def make_failure_qa(question: str, failure_reason: str) -> Dict[str, Any]:
    return {
        "question": question,
        "answer": None,
        "status": "failure",
        "failure_reason": failure_reason,
        "llm_call_count": 0,
        "tool_call_count": 0,
        "usage": {
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
        },
        "runtime_seconds": 0.0,
    }


def run_one_experiment(config: Dict[str, Any], run_id: int) -> Dict[str, Any]:
    dataset_name = config["dataset_name"]
    dataset_path = get_dataset_path(config)
    model_name = get_model_name(config)
    task_type = infer_task_type(config)
    llm_meta = get_llm_metadata()

    output = {
        "baseline_name": "langchain_tool_using_single_agent",
        **llm_meta,
        "run_id": run_id,
        "cost_measurement_mode": "per_question_isolated_agent_run",
        "dataset_name": dataset_name,
        "dataset_path": dataset_path,
        "file_name": config.get("file_name"),
        "background_knowledge": config["background_knowledge"],
        "task_type": task_type,
        "configured_model_name": model_name,
        "configured_model_display_name": MODEL_DISPLAY_NAMES.get(model_name, model_name),
        "x_columns": config["x_columns"],
        "y_column": config["y_column"],
        "questions_and_answers": [],
    }

    if not os.path.exists(dataset_path):
        for question in config["questions"]:
            output["questions_and_answers"].append(
                make_failure_qa(question, f"Dataset not found: {dataset_path}")
            )

        output["dataset_summary"] = summarize_dataset_output(output)
        return output

    try:
        agent = build_agent(config)

        for question in config["questions"]:
            try:
                qa_output = ask_agent(agent, config, question)

            except Exception as exc:
                qa_output = make_failure_qa(question, str(exc))

            output["questions_and_answers"].append(qa_output)

    except Exception as exc:
        for question in config["questions"]:
            output["questions_and_answers"].append(
                make_failure_qa(question, f"Agent construction failed: {str(exc)}")
            )

    output["dataset_summary"] = summarize_dataset_output(output)
    return output


# =========================================================
# 11. Main
# =========================================================

def main():
    if LLM_PROVIDER.lower() == "deepseek":
        if not os.environ.get("DEEPSEEK_API_KEY"):
            raise EnvironmentError("Please set DEEPSEEK_API_KEY.")
        if os.environ.get("DEEPSEEK_API_KEY") == "your_deepseek_api_key_here":
            raise EnvironmentError("Please replace the placeholder DeepSeek API key before running.")

    if LLM_PROVIDER.lower() == "openai":
        if not os.environ.get("OPENAI_API_KEY"):
            raise EnvironmentError("Please set OPENAI_API_KEY.")

    configs = load_experiment_configs(CONFIG_PATH)
    all_outputs = []
    llm_meta = get_llm_metadata()

    for run_id in range(1, NUM_RUNS + 1):
        print("#" * 100)
        print(f"RUN {run_id}/{NUM_RUNS}")
        print("#" * 100)

        for config in configs:
            dataset_name = config["dataset_name"]
            model_name = get_model_name(config)
            task_type = infer_task_type(config)

            print("=" * 100)
            print(f"Running dataset: {dataset_name} | run_id={run_id}")
            print(f"Task type: {task_type}")
            print(f"Configured model: {model_name}")
            print(f"LLM provider: {llm_meta['llm_provider']}")
            print(f"LLM model: {llm_meta['model_id']}")
            print("=" * 100)

            result = run_one_experiment(config, run_id=run_id)
            all_outputs.append(result)

            dataset_output_path = os.path.join(
                OUTPUT_DIR,
                f"run_{run_id}_{safe_name(dataset_name)}_{safe_name(model_name)}_langchain_agent_{safe_name(llm_meta['model_id'])}_outputs.json"
            )

            with open(dataset_output_path, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)

            print("Dataset summary:")
            print(json.dumps(result["dataset_summary"], ensure_ascii=False, indent=2))

            for qa in result["questions_and_answers"]:
                print("-" * 80)
                print(qa["question"])
                print("Status:", qa["status"])
                print("Failure reason:", qa.get("failure_reason"))
                print("LLM calls:", qa["llm_call_count"])
                print("Tool calls:", qa["tool_call_count"])
                print("Prompt tokens:", qa["usage"]["prompt_tokens"])
                print("Completion tokens:", qa["usage"]["completion_tokens"])
                print("Total tokens:", qa["usage"]["total_tokens"])
                print("Runtime seconds:", qa["runtime_seconds"])
                print("Answer:")
                print(qa["answer"])
                print()

            print(f"Saved dataset output to: {dataset_output_path}")

    combined_output = {
        "baseline_name": "langchain_tool_using_single_agent",
        **llm_meta,
        "num_runs": NUM_RUNS,
        "config_path": CONFIG_PATH,
        "outputs": all_outputs,
        "overall_summary": summarize_all_outputs(all_outputs),
        "repeated_run_summary": summarize_repeated_runs(all_outputs),
    }

    combined_output_path = os.path.join(
        OUTPUT_DIR,
        f"all_{safe_name(os.path.splitext(os.path.basename(CONFIG_PATH))[0])}_langchain_agent_{safe_name(llm_meta['model_id'])}_outputs.json"
    )

    with open(combined_output_path, "w", encoding="utf-8") as f:
        json.dump(combined_output, f, ensure_ascii=False, indent=2)

    summary_rows = []
    for output in all_outputs:
        s = output.get("dataset_summary") or {}
        summary_rows.append({
            "baseline_name": output.get("baseline_name"),
            "llm_provider": output.get("llm_provider"),
            "model_display_name": output.get("model_display_name"),
            "model_id": output.get("model_id"),
            "run_id": output.get("run_id"),
            "dataset_name": output.get("dataset_name"),
            "task_type": output.get("task_type"),
            "configured_model_name": output.get("configured_model_name"),
            "num_questions": s.get("num_questions"),
            "success_count": s.get("success_count"),
            "failure_count": s.get("failure_count"),
            "failure_rate": s.get("failure_rate"),
            "total_llm_call_count": s.get("total_llm_call_count"),
            "total_tool_call_count": s.get("total_tool_call_count"),
            "total_prompt_tokens": s.get("total_prompt_tokens"),
            "total_completion_tokens": s.get("total_completion_tokens"),
            "total_tokens": s.get("total_tokens"),
            "total_runtime_seconds": s.get("total_runtime_seconds"),
            "avg_llm_call_count_per_answer": s.get("avg_llm_call_count_per_answer"),
            "avg_tool_call_count_per_answer": s.get("avg_tool_call_count_per_answer"),
            "avg_total_tokens_per_answer": s.get("avg_total_tokens_per_answer"),
            "avg_runtime_seconds_per_answer": s.get("avg_runtime_seconds_per_answer"),
        })

    summary_csv_path = os.path.join(
        OUTPUT_DIR,
        f"summary_{safe_name(os.path.splitext(os.path.basename(CONFIG_PATH))[0])}_langchain_agent_{safe_name(llm_meta['model_id'])}.csv"
    )

    pd.DataFrame(summary_rows).to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

    print("=" * 100)
    print("Overall summary:")
    print(json.dumps(combined_output["overall_summary"], ensure_ascii=False, indent=2))
    print("=" * 100)
    print("Repeated-run summary:")
    print(json.dumps(combined_output["repeated_run_summary"], ensure_ascii=False, indent=2))
    print(f"Saved combined output to: {combined_output_path}")
    print(f"Saved summary CSV to: {summary_csv_path}")


main()